## GPU ENGINE
[Rapids AI open beta Polars post](https://developer.nvidia.com/blog/polars-gpu-engine-powered-by-rapids-cudf-now-available-in-open-beta)

[Google Colab notebook demo from Rapids AI](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/accelerated_data_processing_examples/polars_gpu_engine_demo.ipynb?utm_source=nvidia+mktg&utm_medium=web&utm_campaign=polars+launch)


NOTE: This notebook if for run in Colab or Kaggle. If you have an local environment run the different notebooks of Polars_GPU_XX.ipynb.

In [ ]:
## Only uncomment and run in colab or kaggle to install packages
from IPython.display import clear_output
!pip install uv
!uv pip install "polars[gpu]>=1.27.0" --extra-index-url=https://pypi.nvidia.com
clear_output()

In [ ]:
# Only for Colab. In Kaggle just add dataset: https://www.kaggle.com/datasets/mihaiaftinescu/nyc-green-yellow
!mkdir data
!curl -L -o nyc-green-yellow.zip https://www.kaggle.com/api/v1/datasets/download/mihaiaftinescu/nyc-green-yellow
!unzip nyc-green-yellow.zip -d data/nyc-taxi

In [ ]:
from IPython import get_ipython
import polars as pl
import os

zone_schema = {"LocationID": pl.Int32, "Borough": pl.String, "Zone": pl.String, "service_zone": pl.String}
# Modify path if using kaggle to input dataset
df_zones = pl.scan_csv("data/nyc-taxi/taxi_zone_lookup.csv", schema=zone_schema)

df = (
    pl.scan_parquet("data/nyc-taxi/yellow.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)
df_green = (
    pl.scan_parquet("data/nyc-taxi/green.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)

In [ ]:
df_green.head().collect(engine="streaming")

## Streaming Engine

In [ ]:
%%time
df_green.group_by("trip_distance").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort("trip_distance").head().collect(engine="streaming")

## GPU in-memory
Using the in-memory is the fastest than using managed memory (UVM streaming) but the memory is limited to GPU memory (as it Polars eager).

In [ ]:
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "0"  # Forcing in-memory engine to not use UVM, just for the demo
gpu_in_mem_engine = pl.GPUEngine(executor="in-memory", raise_on_fail=True)

In [ ]:
%%time
# Note: This will fail if GPU memory is not large enough
df_green.group_by("trip_distance").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort("trip_distance").head().collect(engine=gpu_in_mem_engine)

In [ ]:
result, prof = (
    df_green.group_by("trip_distance")
    .agg(
        pl.len().alias("trip_count"),
        pl.sum("fare_amount").alias("total_revenue"),
        pl.mean("fare_amount").alias("avg_fare"),
        pl.mean("trip_distance").alias("avg_distance"),
        pl.mean("tip_amount").alias("avg_tip"),
    )
    .sort("trip_distance")
    .head()
    .profile(engine=gpu_in_mem_engine)
)
prof.with_columns(((pl.col("end") - pl.col("start")) / pl.col("end").max() * 100).alias("pct_time_spent"))

## GPU in memory. More complex queries

In [ ]:
# Recommended to reset kernel to free GPU correctly because some part of the dataset is cached in memory from previous execution
# in the notebook and import again polars import and df definition at the top of the notebook
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
%%time
df_green.group_by(["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"]).head().collect(engine="streaming")

In [ ]:
# Define again gpu in mem engine if kernel restarted
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "0"  # Forcing in-memory engine to not use UVM, just for the demo
gpu_in_mem_engine = pl.GPUEngine(executor="in-memory", raise_on_fail=True)

In [ ]:
%%time
df_green.group_by(["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"]).head().collect(engine=gpu_in_mem_engine)

In [ ]:
result, prof = (
    df_green.group_by(["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"])
    .agg(
        pl.len().alias("trip_count"),
        pl.sum("fare_amount").alias("total_revenue"),
        pl.mean("fare_amount").alias("avg_fare"),
        pl.mean("trip_distance").alias("avg_distance"),
        pl.mean("tip_amount").alias("avg_tip"),
    )
    .sort(["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"])
    .head()
    .profile(engine=gpu_in_mem_engine)
)
prof.with_columns(((pl.col("end") - pl.col("start")) / pl.col("end").max() * 100).alias("pct_time_spent"))

## UVM and streaming for single GPU

Like polars streaming engine, for GPU an streaming memory engine is in beta, aslo taking advantage of UVM (unified virutal memory) combining machine RAM and GPU VRAM.

[UVM, data larger than GPU memory post (NVIDIA blog)](https://developer.nvidia.com/blog/how-to-work-with-data-exceeding-vram-in-the-polars-gpu-engine/)

[UVM, data larger than GPU memory post (Polars blog)](https://pola.rs/posts/uvm-larger-than-ram-gpu/)

[Polars GPU Streaming post (NVIDIA blog)](https://developer.nvidia.com/blog/rapids-adds-gpu-polars-streaming-a-unified-gnn-api-and-zero-code-ml-speedups)


In [ ]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(
    engine=gpu_in_mem_engine
)  # RuntimeError: parallel_for failed: cudaErrorMemoryAllocation: out of memory

In [ ]:
# Recommended to reset kernel to free GPU correctly because some part of the dataset is cached in memory from previous execution
# in the notebook and import again polars import and df definition at the top of the notebook
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
# This is the manual configuration released in the first UVM post on polars and NVIDIA post.
import rmm

mr = rmm.mr.PrefetchResourceAdaptor(rmm.mr.PoolMemoryResource(rmm.mr.ManagedMemoryResource()))
gpu_streaming_engine = pl.GPUEngine(memory_resource=mr, raise_on_fail=True)


# The new simplified version is passing streaming executor to the GPU engine. This is the default setting if no executor is specified
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "1" # This is the default config
gpu_streaming_engine = pl.GPUEngine(executor="streaming", raise_on_fail=True)

In [ ]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=gpu_streaming_engine)

In [ ]:
# Recommended to reset kernel to free GPU correctly because some part of the dataset is cached in memory from previous execution
# in the notebook and import again polars import and df definition at the top of the notebook
get_ipython().kernel.do_shutdown(restart=True)

## Multi GPU Streaming. Currently in experimental stage.
#### NOTE: Only run this part in an environment where are multiple GPUs. Kaggle can be a free option to test it.

In [ ]:
!uv pip install dask_cuda

In [ ]:
## Multi GPU Streaming
from dask_cuda import LocalCUDACluster
from dask.distributed import Client

client = Client(LocalCUDACluster())

multi_gpu_streaming = pl.GPUEngine(
  executor="streaming",
  executor_options={"scheduler": "distributed"} # For a single machine, the executor option is synchronous
)

In [ ]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=multi_gpu_streaming)